## 1. Imports

In [ ]:
import os
import csv
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from IPython.display import display, HTML

# Base paths
BASE_TRACKING_PATH = Path("tracking")
BASE_HEATMAPS_PATH = Path("heatmaps")
BASE_BUBBLE_PLOTS_PATH = Path("bubble_plots")
BASE_ROSE_DIAGRAMS_PATH = Path("rose_diagrams")

## 2. Configuration and constants

In [ ]:
# ============================================================================
# CONFIGURATION CONSTANTS (consistent with the kinematics notebook)
# ============================================================================

SCALE = 0.2  # mm/px - pixel-to-millimeter conversion factor
PLATE_SIZE_MM = 95  # mm - Petri dish size
PLATE_SIZE_PX = int(PLATE_SIZE_MM / SCALE)  # ~475 px

# Artifact filtering (consistent with the kinematics)
MIN_MOVEMENT_THRESHOLD_MM = 0.5  # mm - min movement threshold (filters detection jitter)
MAX_MOVEMENT_THRESHOLD_MM = 5.0  # mm - max movement threshold (filters artifacts/jumps)
MAX_MOVEMENT_THRESHOLD_PX = MAX_MOVEMENT_THRESHOLD_MM / SCALE  # 25 px
ARTIFACT_PROXIMITY = 5  # frames - movements near artifacts are ignored

In [ ]:
# ============================================================================
# MAPPING DICTIONARIES
# ============================================================================

SESSIONS = {
    1: "01",
    2: "02"
}

# Experiments for session 01
EXPERIMENTS_01 = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x_10_7",
    5: "coli_5x_10_8"
}

# Experiments for session 02
EXPERIMENTS_02 = {
    1: "control",
    2: "pbs",
    3: "coli_2x10_8",
    4: "coli_5x10_7",
    5: "coli_5x10_8"
}

# Mapping of experiment names to display names
EXPERIMENT_DISPLAY_NAMES = {
    "control": "Control (no injection)",
    "naklucie": "Injection",
    "pbs": "PBS",
    "coli_2x10_8": "E. coli 2x10^8",
    "coli_5x_10_7": "E. coli 5x10^7",
    "coli_5x_10_8": "E. coli 5x10^8",
    "coli_5x10_7": "E. coli 5x10^7",
    "coli_5x10_8": "E. coli 5x10^8"
}

In [ ]:
# ============================================================================
# DATA-SELECTION FUNCTIONS
# ============================================================================

def display_sessions():
    """Print the available sessions."""
    print("\n=== Available sessions ===")
    for num, name in SESSIONS.items():
        print(f"{num}. Session {name}")
    print("======================\n")

def display_experiments(session):
    """Print the available experiments for a session."""
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02
    print(f"\n=== Experiments for session {session} ===")
    for num, name in experiments.items():
        print(f"{num}. {name}")
    print("="*35 + "\n")

def get_session():
    """Read the session number from the user."""
    display_sessions()
    while True:
        try:
            session_num = int(input('Enter session number (1-2): '))
            if session_num in SESSIONS:
                return SESSIONS[session_num]
            else:
                print('Error: choose 1 or 2')
        except ValueError:
            print('Error: enter an integer')

def get_experiment(session):
    """Read the experiment from the user."""
    experiments = EXPERIMENTS_01 if session == "01" else EXPERIMENTS_02
    display_experiments(session)
    while True:
        try:
            exp_num = int(input(f'Enter experiment number (1-{len(experiments)}): '))
            if exp_num in experiments:
                return experiments[exp_num]
            else:
                print(f'Error: choose a number between 1 and {len(experiments)}')
        except ValueError:
            print('Error: enter an integer')

In [ ]:
# ============================================================================
# HELPER FUNCTIONS - PATHS AND FILTERING
# ============================================================================

def get_larva_file(session: str, experiment: str, larva_num: int) -> Optional[Path]:
    """
    Return the path to a larva file.

    NOTE: files come in different formats!
    - larva_positions_X.csv -> data in PIXELS (X=464, Y=185)
    - larva_positions_X_with_angles.csv -> data in MM (X=92.8, Y=37.0), scaled by SCALE
    - larva_positions_X_cleaned.csv -> data in PIXELS

    We prefer px files for consistency.
    """
    exp_path = BASE_TRACKING_PATH / session / experiment
    
    # File preference order
    file_options = [
        exp_path / f'larva_positions_{larva_num}_cleaned.csv',   # px (cleaned)
        exp_path / f'larva_positions_{larva_num}.csv',           # px (basic)
        exp_path / f'larva_positions_{larva_num}_with_angles.csv'  # mm (fallback)
    ]
    
    for file_path in file_options:
        if file_path.exists():
            return file_path
    
    return None

def get_larva_file_with_angles(session: str, experiment: str, larva_num: int) -> Optional[Path]:
    """
    Return the path to a larva file WITH ANGLES (for rose diagrams).

    Prefers files with an Angle column:
    - larva_positions_X_with_angles.csv -> data in MM + Angle, Distance columns

    Returns:
        Path to the file, or None if not found.
    """
    exp_path = BASE_TRACKING_PATH / session / experiment
    
    # Rose diagrams need a file with angles
    file_with_angles = exp_path / f'larva_positions_{larva_num}_with_angles.csv'
    
    if file_with_angles.exists():
        return file_with_angles
    
    return None

def detect_data_unit(x_values: List[float], y_values: List[float]) -> str:
    """
    Detect whether data is in pixels or millimeters.

    Logic:
    - a Petri dish is 95 mm = 475 px (at scale 0.2 mm/px)
    - if max(X,Y) > 200 -> pixels
    - if max(X,Y) < 200 -> millimeters

    Returns:
        'px' or 'mm'
    """
    if len(x_values) == 0:
        return 'px'
    
    max_val = max(max(x_values), max(y_values))
    
    # A dish is ~95 mm ~= 475 px; values > 200 are certainly pixels
    if max_val > 200:
        return 'px'
    else:
        return 'mm'

def get_output_dir(base_path: Path, session: str, experiment: str) -> Path:
    """Return the output folder for the current session/experiment."""
    output_dir = base_path / session / experiment
    output_dir.mkdir(parents=True, exist_ok=True)
    return output_dir

def filter_artifacts(x_positions: List[float], y_positions: List[float], data_unit: str) -> Tuple[List[float], List[float], int]:
    """
    Filter artifacts from positions (consistent with the kinematics notebook).

    Filtering:
    - movements > MAX_MOVEMENT_THRESHOLD (5 mm = 25 px) = artifacts
    - positions near artifacts (+/-5 frames) = related to multi-detection

    Args:
        x_positions, y_positions: position lists
        data_unit: 'px' or 'mm'

    Returns:
        Tuple: (filtered_x, filtered_y, number_of_removed_artifacts)
    """
    if len(x_positions) < 2:
        return x_positions, y_positions, 0
    
    # Threshold in the appropriate units
    if data_unit == 'px':
        threshold = MAX_MOVEMENT_THRESHOLD_PX  # 25px
    else:
        threshold = MAX_MOVEMENT_THRESHOLD_MM  # 5mm
    
    # Find artifact indices (jumps > threshold)
    artifact_indices = set()
    for i in range(len(x_positions) - 1):
        dx = x_positions[i + 1] - x_positions[i]
        dy = y_positions[i + 1] - y_positions[i]
        dist = np.sqrt(dx**2 + dy**2)
        
        if dist > threshold:
            artifact_indices.add(i)
            artifact_indices.add(i + 1)
    
    # Expand to neighboring frames (ARTIFACT_PROXIMITY)
    expanded_artifacts = set()
    for idx in artifact_indices:
        for offset in range(-ARTIFACT_PROXIMITY, ARTIFACT_PROXIMITY + 1):
            new_idx = idx + offset
            if 0 <= new_idx < len(x_positions):
                expanded_artifacts.add(new_idx)
    
    # Filter positions
    filtered_x = []
    filtered_y = []
    for i in range(len(x_positions)):
        if i not in expanded_artifacts:
            filtered_x.append(x_positions[i])
            filtered_y.append(y_positions[i])
    
    return filtered_x, filtered_y, len(expanded_artifacts)

In [ ]:
# ============================================================================
# DATA SELECTION - RUN THIS CELL
# ============================================================================

# Select session
SESSION = get_session()

# Select experiment
EXPERIMENT = get_experiment(SESSION)

# Summary
print("\n" + "="*50)
print("SELECTED:")
print(f"  Session: {SESSION}")
print(f"  Experiment: {EXPERIMENT}")
print(f"  Name: {EXPERIMENT_DISPLAY_NAMES.get(EXPERIMENT, EXPERIMENT)}")
print(f"  Output folders:")
print(f"    - Heatmaps: heatmaps/{SESSION}/{EXPERIMENT}/")
print(f"    - Bubble plots: bubble_plots/{SESSION}/{EXPERIMENT}/")
print(f"    - Rose diagrams: rose_diagrams/{SESSION}/{EXPERIMENT}/")
print("="*50)

## 3. Heatmap (most frequently visited positions)

**NOTE:** The heatmap shows the **frequency** of visits (how many times the larva was at a given location), not just coverage.

- **% coverage** = how many grid cells have at least 1 visit -> "where the larva was"
- **Heatmap** = how many times the larva was in a given cell -> "where the larva spent time"

So a larva may have 33% coverage but the heatmap shows mostly the edges - because it spent 90% of the time there (thigmotaxis).

In [ ]:
# ============================================================================
# COMBINED HEATMAP CLASS
# ============================================================================

class CombinedHeatmapPlotter:
    """
    Generates combined heatmap plots for an experiment group.
    Creates one image with 6 plots (for 6 larvae) - a 2x3 grid.

    Includes artifact filtering consistent with the kinematics notebook.
    Automatically detects data units (px vs mm).

    Scale options:
    - Linear: high-frequency areas dominate (edges)
    - Logarithmic: rarely visited areas are more visible (dish center)
    """
    
    def __init__(self, session: str, experiment: str):
        self.session = session
        self.experiment = experiment
        self.display_name = EXPERIMENT_DISPLAY_NAMES.get(experiment, experiment)
        self.tracking_dir = BASE_TRACKING_PATH / session / experiment
        self.output_dir = get_output_dir(BASE_HEATMAPS_PATH, session, experiment)
        
        # Data for each larva
        self.larvae_data: Dict[int, Dict] = {}
    
    def _read_csv(self, larva_num: int) -> Tuple[List[float], List[float], str]:
        """Load position data from a CSV file."""
        file_path = get_larva_file(self.session, self.experiment, larva_num)
        
        if file_path is None:
            return [], [], "missing"
        
        file_type = "standard"
        if "_cleaned" in file_path.name:
            file_type = "cleaned"
        elif "_with_angles" in file_path.name:
            file_type = "with_angles"
        
        x_positions = []
        y_positions = []
        
        with open(file_path, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                x_positions.append(float(row['X']))
                y_positions.append(float(row['Y']))
        
        return x_positions, y_positions, file_type
    
    def _calculate_coverage(self, x_positions: List[float], y_positions: List[float], data_unit: str) -> float:
        """
        Compute the dish coverage percentage.

        Logic:
        - a dish is 95 mm = 475 px (at scale 0.2 mm/px)
        - we split the dish into a 100x100 grid of bins
        - count visited cells / total number of cells

        Args:
            x_positions, y_positions: position lists
            data_unit: 'px' or 'mm'
        """
        if len(x_positions) == 0:
            return 0.0
        
        # Use 100 bins - a 100x100 grid
        n_bins = 100
        total_cells = n_bins * n_bins  # 10,000 cells
        
        # Range depends on the data unit
        if data_unit == 'px':
            max_range = PLATE_SIZE_PX  # 475 px
        else:
            max_range = PLATE_SIZE_MM  # 95 mm
        
        # Compute the 2D histogram
        hist, _, _ = np.histogram2d(
            x_positions, 
            y_positions, 
            bins=n_bins,
            range=[[0, max_range], [0, max_range]]
        )
        
        # Number of non-empty (visited) cells
        visited_cells = np.count_nonzero(hist)
        
        # Coverage percentage
        coverage = (visited_cells / total_cells) * 100
        
        return coverage
    
    def load_all_larvae(self) -> None:
        """Load data for all 6 larvae with artifact filtering."""
        print(f"Loading data for: {self.display_name} (session {self.session})")
        print(f"   Path: {self.tracking_dir}")
        print(f"   Filtering: MAX_MOVEMENT > {MAX_MOVEMENT_THRESHOLD_MM}mm ({MAX_MOVEMENT_THRESHOLD_PX:.0f}px), proximity: +/-{ARTIFACT_PROXIMITY} frames")
        
        for larva_num in range(1, 7):
            x_raw, y_raw, file_type = self._read_csv(larva_num)
            
            if x_raw and y_raw:
                original_count = len(x_raw)
                
                # Detect the data unit (px vs mm)
                data_unit = detect_data_unit(x_raw, y_raw)
                
                # Filter artifacts (taking units into account)
                x_filtered, y_filtered, artifacts_removed = filter_artifacts(x_raw, y_raw, data_unit)
                
                # Compute coverage (taking units into account)
                coverage = self._calculate_coverage(x_filtered, y_filtered, data_unit)
                
                self.larvae_data[larva_num] = {
                    'x': x_filtered,
                    'y': y_filtered,
                    'data_unit': data_unit,
                    'coverage': coverage,
                    'original_count': original_count,
                    'filtered_count': len(x_filtered),
                    'artifacts_removed': artifacts_removed,
                    'file_type': file_type
                }
                
                tag = f" [{file_type.upper()}]" if file_type != "standard" else ""
                print(f"  [ok] Larva {larva_num}: {original_count} -> {len(x_filtered)} pts (-{artifacts_removed} artifacts), coverage: {coverage:.2f}% [{data_unit}]{tag}")
            else:
                self.larvae_data[larva_num] = None
                print(f"  [warn] Larva {larva_num}: no file")
    
    def _plot_single_heatmap(self, ax, larva_num: int, use_log_scale: bool = True) -> dict:
        """
        Draw a single larva's heatmap on the given subplot.

        Args:
            ax: subplot
            larva_num: larva number
            use_log_scale: whether to use a log scale (makes rare areas more visible)

        Returns:
            dict with plot info (min_count, max_count, im) or None.
        """
        data = self.larvae_data.get(larva_num)
        
        if data is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                    fontsize=14, transform=ax.transAxes, color='red')
            ax.set_title(f'Larva {larva_num}', fontsize=12, fontweight='bold')
            ax.set_xlim(0, PLATE_SIZE_PX)
            ax.set_ylim(PLATE_SIZE_PX, 0)
            ax.set_facecolor('black')
            ax.set_aspect('equal', adjustable='box')
            return None
        
        x, y = data['x'], data['y']
        coverage = data['coverage']
        data_unit = data['data_unit']
        
        # Set the range based on the unit
        if data_unit == 'px':
            max_range = PLATE_SIZE_PX
            unit_label = 'px'
        else:
            max_range = PLATE_SIZE_MM
            unit_label = 'mm'
        
        # Compute the 2D histogram manually for control over normalization
        hist, xedges, yedges = np.histogram2d(
            x, y,
            bins=100,
            range=[[0, max_range], [0, max_range]]
        )
        
        # Transpose - histogram2d returns (x, y), imshow expects (y, x)
        hist = hist.T
        
        # Statistics
        max_count = int(hist.max())
        non_zero_hist = hist[hist > 0]
        min_count = int(non_zero_hist.min()) if len(non_zero_hist) > 0 else 0
        
        # Log scale - rarely visited areas are more visible
        if use_log_scale and max_count > 0:
            # log1p = log(1+x) - avoids log(0), keeps 0 as 0
            hist_display = np.log1p(hist)
        else:
            hist_display = hist
        
        # Draw the heatmap
        im = ax.imshow(
            hist_display,
            extent=[0, max_range, max_range, 0],  # [left, right, bottom, top] - inverted Y axis
            cmap='hot',
            aspect='equal',
            interpolation='nearest'
        )
        
        # Title with coverage percentage and max visits
        ax.set_title(f'Larva {larva_num}\n{coverage:.2f}% coverage (max: {max_count})', fontsize=10, fontweight='bold')
        
        ax.set_xlim(0, max_range)
        ax.set_ylim(max_range, 0)
        ax.set_xlabel(f'X ({unit_label})', fontsize=9)
        ax.set_ylabel(f'Y ({unit_label})', fontsize=9)
        ax.tick_params(axis='both', labelsize=8)
        
        return {'min_count': min_count, 'max_count': max_count, 'im': im, 'hist': hist}
    
    def plot_combined(self, save: bool = True, use_log_scale: bool = True) -> None:
        """
        Create the combined heatmap plot (2x3 grid).

        Args:
            save: whether to save to a file
            use_log_scale: whether to use a log scale
                - True: rare areas are more visible (dish center)
                - False: linear scale, most-visited areas dominate (edges)
        """
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        
        scale_label = "log scale" if use_log_scale else "linear scale"
        fig.suptitle(f'{self.display_name} - Position heatmaps ({scale_label}) - session {self.session}', 
                     fontsize=16, fontweight='bold', y=0.98)
        
        axes_flat = axes.flatten()
        plot_info = []
        global_max = 0
        
        for idx, larva_num in enumerate(range(1, 7)):
            info = self._plot_single_heatmap(axes_flat[idx], larva_num, use_log_scale)
            if info:
                plot_info.append(info)
                global_max = max(global_max, info['max_count'])
        
        # Colorbar - use the last image as reference
        if plot_info:
            fig.subplots_adjust(bottom=0.12)
            cbar_ax = fig.add_axes([0.15, 0.04, 0.7, 0.02])
            cbar = fig.colorbar(plot_info[-1]['im'], cax=cbar_ax, orientation='horizontal')
            
            if use_log_scale:
                cbar.set_label(f'log(1 + visit count) | max visits in group: {global_max}', fontsize=10)
            else:
                cbar.set_label(f'Visit count (frames per bin) | max: {global_max}', fontsize=10)
        
        plt.tight_layout(rect=[0, 0.08, 1, 0.96])
        
        if save:
            suffix = "_log" if use_log_scale else "_linear"
            output_path = self.output_dir / f"combined_heatmaps{suffix}.png"
            plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
            print(f"\n  Saved: {output_path}")
        
        plt.show()
    
    def run(self, use_log_scale: bool = True) -> None:
        """Run the full pipeline."""
        self.load_all_larvae()
        print()
        self.plot_combined(use_log_scale=use_log_scale)
        print("\n" + "="*60 + "\n")

In [ ]:
# ============================================================================
# GENERATE HEATMAPS
# ============================================================================

# use_log_scale=True  -> rare areas are more visible (dish center)
# use_log_scale=False -> linear scale, most-visited areas dominate (edges, thigmotaxis)

heatmap_plotter = CombinedHeatmapPlotter(SESSION, EXPERIMENT)
heatmap_plotter.run(use_log_scale=True)

In [ ]:
# For comparison - also generate the linear-scale version
heatmap_plotter.plot_combined(use_log_scale=False)

## 4. Bubble plot (most frequently visited positions)

In [ ]:
# ============================================================================
# COMBINED BUBBLE PLOT CLASS
# ============================================================================

class CombinedBubblePlotter:
    """
    Generates combined bubble plots for an experiment group.
    Creates one image with 6 plots (for 6 larvae) - a 2x3 grid.

    Includes artifact filtering consistent with the kinematics notebook.
    """
    
    def __init__(self, session: str, experiment: str, n_top_locations: int = 100):
        self.session = session
        self.experiment = experiment
        self.n_top = n_top_locations
        self.display_name = EXPERIMENT_DISPLAY_NAMES.get(experiment, experiment)
        self.tracking_dir = BASE_TRACKING_PATH / session / experiment
        self.output_dir = get_output_dir(BASE_BUBBLE_PLOTS_PATH, session, experiment)
        
        # Data for each larva
        self.larvae_data: Dict[int, Dict] = {}
    
    def _read_csv(self, larva_num: int) -> Tuple[List[float], List[float], str]:
        """Load position data from a CSV file."""
        file_path = get_larva_file(self.session, self.experiment, larva_num)
        
        if file_path is None:
            return [], [], "missing"
        
        file_type = "standard"
        if "_cleaned" in file_path.name:
            file_type = "cleaned"
        elif "_with_angles" in file_path.name:
            file_type = "with_angles"
        
        x_positions = []
        y_positions = []
        
        with open(file_path, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                x_positions.append(float(row['X']))
                y_positions.append(float(row['Y']))
        
        return x_positions, y_positions, file_type
    
    def load_all_larvae(self) -> None:
        """Load data for all 6 larvae with artifact filtering."""
        print(f"Loading data for: {self.display_name} (session {self.session})")
        print(f"   Path: {self.tracking_dir}")
        print(f"   Filtering: MAX_MOVEMENT > {MAX_MOVEMENT_THRESHOLD_MM}mm ({MAX_MOVEMENT_THRESHOLD_PX:.0f}px), proximity: +/-{ARTIFACT_PROXIMITY} frames")
        
        for larva_num in range(1, 7):
            x_raw, y_raw, file_type = self._read_csv(larva_num)
            
            if x_raw and y_raw:
                original_count = len(x_raw)
                
                # Detect the data unit
                data_unit = detect_data_unit(x_raw, y_raw)
                
                # Filtruj artefakty
                x_filtered, y_filtered, artifacts_removed = filter_artifacts(x_raw, y_raw, data_unit)
                
                # Build a DataFrame and group
                df = pd.DataFrame({'X': x_filtered, 'Y': y_filtered})
                # Round for grouping - 1px for px, 0.2mm for mm
                if data_unit == 'px':
                    df['X'] = df['X'].round(0)
                    df['Y'] = df['Y'].round(0)
                else:
                    df['X'] = df['X'].round(1)
                    df['Y'] = df['Y'].round(1)
                
                grouped = df.groupby(['X', 'Y']).size().reset_index(name='count')
                top_n = grouped.nlargest(self.n_top, 'count')
                
                self.larvae_data[larva_num] = {
                    'top_locations': top_n,
                    'data_unit': data_unit,
                    'original_count': original_count,
                    'filtered_count': len(x_filtered),
                    'artifacts_removed': artifacts_removed,
                    'file_type': file_type
                }
                
                tag = f" [{file_type.upper()}]" if file_type != "standard" else ""
                print(f"  [ok] Larva {larva_num}: {original_count} -> {len(x_filtered)} pts (-{artifacts_removed} artifacts) [{data_unit}]{tag}")
            else:
                self.larvae_data[larva_num] = None
                print(f"  [warn] Larva {larva_num}: no file")
    
    def _plot_single_bubble(self, ax, larva_num: int) -> None:
        """Draw a single larva's bubble plot on the given subplot."""
        data = self.larvae_data.get(larva_num)
        
        if data is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                    fontsize=14, transform=ax.transAxes, color='red')
            ax.set_title(f'Larva {larva_num}', fontsize=12, fontweight='bold')
            ax.set_xlim(0, PLATE_SIZE_PX)
            ax.set_ylim(PLATE_SIZE_PX, 0)
            ax.set_aspect('equal', adjustable='box')
            ax.grid(True, alpha=0.3)
            return
        
        top_locations = data['top_locations']
        data_unit = data['data_unit']
        
        # Set the range based on the unit
        if data_unit == 'px':
            max_range = PLATE_SIZE_PX
            unit_label = 'px'
        else:
            max_range = PLATE_SIZE_MM
            unit_label = 'mm'
        
        # Draw the bubble plot
        ax.scatter(
            top_locations['X'], 
            top_locations['Y'], 
            s=top_locations['count'] * 5,
            c='blue', 
            alpha=0.5, 
            edgecolors='k',
            linewidths=0.5
        )
        
        ax.set_title(f'Larva {larva_num}\n(top {self.n_top} locations)', fontsize=11, fontweight='bold')
        ax.set_xlim(0, max_range)
        ax.set_ylim(max_range, 0)
        ax.set_aspect('equal', adjustable='box')
        ax.set_xlabel(f'X ({unit_label})', fontsize=9)
        ax.set_ylabel(f'Y ({unit_label})', fontsize=9)
        ax.tick_params(axis='both', labelsize=8)
        ax.grid(True, alpha=0.3)
    
    def plot_combined(self, save: bool = True) -> None:
        """Create the combined bubble plot (2x3 grid)."""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        fig.suptitle(f'{self.display_name} - Most frequently visited positions (Bubble) - session {self.session}', 
                     fontsize=16, fontweight='bold', y=0.98)
        
        axes_flat = axes.flatten()
        
        for idx, larva_num in enumerate(range(1, 7)):
            self._plot_single_bubble(axes_flat[idx], larva_num)
        
        # Shared legend
        fig.text(0.5, 0.02, 'Bubble size is proportional to visit frequency', 
                 ha='center', fontsize=11, style='italic')
        
        plt.tight_layout(rect=[0, 0.05, 1, 0.96])
        
        if save:
            output_path = self.output_dir / f"combined_bubble_top_{self.n_top}.png"
            plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
            print(f"\n  Saved: {output_path}")
        
        plt.show()
    
    def run(self) -> None:
        """Run the full pipeline."""
        self.load_all_larvae()
        print()
        self.plot_combined()
        print("\n" + "="*60 + "\n")

In [ ]:
# ============================================================================
# GENERATE BUBBLE PLOTS
# ============================================================================

# Number of top visited locations to display
N_TOP_LOCATIONS = 100

bubble_plotter = CombinedBubblePlotter(SESSION, EXPERIMENT, N_TOP_LOCATIONS)
bubble_plotter.run()

## 5. Rose diagram (movement directions)

In [ ]:
# ============================================================================
# COMBINED ROSE DIAGRAM CLASS
# ============================================================================

class CombinedRoseDiagramPlotter:
    """
    Generates combined rose diagrams for an experiment group.
    Creates one image with 6 plots (for 6 larvae) - a 2x3 grid.

    Includes artifact filtering consistent with the kinematics notebook.
    """
    
    def __init__(self, session: str, experiment: str, bin_size: int = 15):
        self.session = session
        self.experiment = experiment
        self.bin_size = bin_size
        self.n_bins = int(360 / bin_size)
        self.display_name = EXPERIMENT_DISPLAY_NAMES.get(experiment, experiment)
        self.tracking_dir = BASE_TRACKING_PATH / session / experiment
        self.output_dir = get_output_dir(BASE_ROSE_DIAGRAMS_PATH, session, experiment)
        
        # Data for each larva
        self.larvae_data: Dict[int, Dict] = {}
    
    def _read_csv_with_angles(self, larva_num: int) -> Tuple[List[float], List[float], List[float], List[float], str]:
        """Load position, angle and distance data from a CSV file (requires the _with_angles file)."""
        # Rose diagrams use a dedicated helper that looks for files with angles
        file_path = get_larva_file_with_angles(self.session, self.experiment, larva_num)
        
        if file_path is None:
            return [], [], [], [], "missing"
        
        file_type = "with_angles"
        
        x_positions = []
        y_positions = []
        angles = []
        distances = []
        
        with open(file_path, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                x_positions.append(float(row['X']))
                y_positions.append(float(row['Y']))
                if 'Angle' in row:
                    angles.append(float(row['Angle']))
                if 'Distance' in row:
                    distances.append(float(row['Distance']))
        
        return x_positions, y_positions, angles, distances, file_type
    
    def _filter_angles_by_distance(self, angles: List[float], distances: List[float], 
                                   min_distance: float = MIN_MOVEMENT_THRESHOLD_MM) -> Tuple[List[float], int, int]:
        """
        Filter angles based on the minimum movement distance.

        KEY POINT: angles at Distance=0 or very small distances are detection noise,
        not real larva movement. arctan2(0,0) returns 0, which falsely inflates the N direction.

        Args:
            angles: list of angles
            distances: list of distances
            min_distance: minimum distance in mm (default MIN_MOVEMENT_THRESHOLD_MM = 0.5mm)

        Returns:
            Tuple: (filtered_angles, jitter_removed, artifacts_removed)
        """
        if len(angles) == 0 or len(distances) == 0:
            return angles, 0, 0
        
        if len(angles) != len(distances):
            print(f"    [warn] Length mismatch: angles={len(angles)}, distances={len(distances)}")
            return angles, 0, 0
        
        filtered_angles = []
        jitter_removed = 0
        artifacts_removed = 0
        
        for i, (angle, dist) in enumerate(zip(angles, distances)):
            # Filter movements that are too small (detection jitter)
            if dist < min_distance:
                jitter_removed += 1
                continue
            
            # Filter movements that are too large (artifacts/jumps)
            if dist > MAX_MOVEMENT_THRESHOLD_MM:
                artifacts_removed += 1
                continue
            
            filtered_angles.append(angle)
        
        return filtered_angles, jitter_removed, artifacts_removed
    
    def load_all_larvae(self) -> None:
        """Load data for all 6 larvae with noise and artifact filtering."""
        print(f"Loading data for: {self.display_name} (session {self.session})")
        print(f"   Path: {self.tracking_dir}")
        print(f"   Filtering: {MIN_MOVEMENT_THRESHOLD_MM}mm <= Distance <= {MAX_MOVEMENT_THRESHOLD_MM}mm")
        print(f"   (removes detection jitter and artifacts/jumps)")
        
        for larva_num in range(1, 7):
            x_vals, y_vals, angles, distances, file_type = self._read_csv_with_angles(larva_num)
            
            if x_vals and y_vals and angles and distances:
                original_count = len(angles)
                
                # Filter angles based on distance (removes jitter and artifacts)
                filtered_angles, jitter_removed, artifacts_removed = self._filter_angles_by_distance(
                    angles, distances, MIN_MOVEMENT_THRESHOLD_MM
                )
                
                self.larvae_data[larva_num] = {
                    'angles': filtered_angles,
                    'original_count': original_count,
                    'filtered_count': len(filtered_angles),
                    'jitter_removed': jitter_removed,
                    'artifacts_removed': artifacts_removed,
                    'file_type': file_type
                }
                
                print(f"  [ok] Larva {larva_num}: {original_count} -> {len(filtered_angles)} angles (-{jitter_removed} jitter, -{artifacts_removed} artifacts)")
            elif x_vals and y_vals and angles and not distances:
                self.larvae_data[larva_num] = None
                print(f"  [warn] Larva {larva_num}: no 'Distance' column in file")
            elif x_vals and y_vals and not angles:
                self.larvae_data[larva_num] = None
                print(f"  [warn] Larva {larva_num}: no 'Angle' column in file")
            else:
                self.larvae_data[larva_num] = None
                print(f"  [warn] Larva {larva_num}: no _with_angles.csv file (run the angle processor notebook)")
    
    def _plot_single_rose(self, ax, larva_num: int) -> None:
        """Draw a single larva's rose diagram on the given subplot."""
        data = self.larvae_data.get(larva_num)
        
        if data is None:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', 
                    fontsize=14, transform=ax.transAxes, color='red')
            ax.set_title(f'Larva {larva_num}', fontsize=12, fontweight='bold')
            return
        
        angles = data['angles']
        angles_radians = np.deg2rad(angles)
        
        # Compute the angle histogram
        counts, bins = np.histogram(
            angles_radians, 
            bins=np.linspace(
                -np.deg2rad(self.bin_size)/2, 
                2*np.pi - np.deg2rad(self.bin_size)/2, 
                self.n_bins + 1
            )
        )
        
        # Draw the rose diagram
        ax.bar(
            bins[:-1], 
            counts, 
            width=np.deg2rad(self.bin_size), 
            edgecolor='k',
            align='edge',
            alpha=0.7
        )
        
        # Put N at the top, clockwise direction
        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)
        
        ax.set_title(f'Larva {larva_num}\n({len(angles)} observations)', fontsize=11, fontweight='bold', pad=10)
    
    def plot_combined(self, save: bool = True) -> None:
        """Create the combined rose diagram (2x3 grid)."""
        fig, axes = plt.subplots(2, 3, figsize=(15, 10), subplot_kw={'projection': 'polar'})
        fig.suptitle(f'{self.display_name} - Movement direction distribution (Rose Diagram) - session {self.session}', 
                     fontsize=16, fontweight='bold', y=0.98)
        
        axes_flat = axes.flatten()
        
        for idx, larva_num in enumerate(range(1, 7)):
            self._plot_single_rose(axes_flat[idx], larva_num)
        
        # Shared legend
        fig.text(0.5, 0.02, f'Bin size: {self.bin_size} deg | N = top', 
                 ha='center', fontsize=11, style='italic')
        
        plt.tight_layout(rect=[0, 0.05, 1, 0.96])
        
        if save:
            output_path = self.output_dir / f"combined_rose_diagrams_{self.bin_size}deg.png"
            plt.savefig(output_path, dpi=150, bbox_inches='tight', facecolor='white')
            print(f"\n  Saved: {output_path}")
        
        plt.show()
    
    def run(self) -> None:
        """Run the full pipeline."""
        self.load_all_larvae()
        print()
        self.plot_combined()
        print("\n" + "="*60 + "\n")

In [ ]:
# ============================================================================
# GENERATE ROSE DIAGRAMS
# ============================================================================

# Angular bin size (in degrees)
BIN_SIZE = 15

rose_plotter = CombinedRoseDiagramPlotter(SESSION, EXPERIMENT, BIN_SIZE)
rose_plotter.run()

## 6. Generate all plots at once

In [ ]:
# ============================================================================
# GENERATE ALL PLOTS AT ONCE
# ============================================================================

def generate_all_plots(session: str, experiment: str, n_top: int = 100, bin_size: int = 15):
    """Generate all plot types for the selected session/experiment."""
    display_name = EXPERIMENT_DISPLAY_NAMES.get(experiment, experiment)
    
    print("\n" + "#"*60)
    print("#" + " "*58 + "#")
    print("#" + "    GENERATING ALL PLOTS".center(58) + "#")
    print("#" + f"    Session: {session} | Experiment: {display_name}".center(58) + "#")
    print("#" + " "*58 + "#")
    print("#"*60 + "\n")
    
    # Heatmaps (both scales)
    print("\n" + "="*60)
    print("HEATMAPS")
    print("="*60)
    heatmap_plotter = CombinedHeatmapPlotter(session, experiment)
    heatmap_plotter.load_all_larvae()
    print("\n--- Log scale ---")
    heatmap_plotter.plot_combined(use_log_scale=True)
    print("\n--- Linear scale ---")
    heatmap_plotter.plot_combined(use_log_scale=False)
    
    # Bubble plots
    print("\n" + "="*60)
    print("BUBBLE PLOTS")
    print("="*60)
    bubble_plotter = CombinedBubblePlotter(session, experiment, n_top)
    bubble_plotter.run()
    
    # Rose diagrams
    print("\n" + "="*60)
    print("ROSE DIAGRAMS")
    print("="*60)
    rose_plotter = CombinedRoseDiagramPlotter(session, experiment, bin_size)
    rose_plotter.run()
    
    print("\n" + "#"*60)
    print("#" + " "*58 + "#")
    print("#" + "    ALL PLOTS GENERATED!".center(58) + "#")
    print("#" + " "*58 + "#")
    print("#"*60)
    
    print(f"\nFiles saved in:")
    print(f"  - Heatmaps:      heatmaps/{session}/{experiment}/")
    print(f"  - Bubble plots:  bubble_plots/{session}/{experiment}/")
    print(f"  - Rose diagrams: rose_diagrams/{session}/{experiment}/")

# Uncomment the line below to generate all plots at once
# generate_all_plots(SESSION, EXPERIMENT, N_TOP_LOCATIONS, BIN_SIZE)